# 파이널 테스트
모범 답변과 챗봇 답변을 Ragas로 비교하는 코드입니다. \
Similarity를 기본 점수로 설정하고 Correctness를 가산점/감점으로 설정하여 비교합니다. \
\
아래 코드는 GotQuestions 데이터를 기준으로 만들어졌습니다. 파일명과 컬럼명 수정이 필요합니다.

In [6]:
import pandas as pd
import logging
from dotenv import load_dotenv
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import answer_correctness, answer_similarity
from langchain_upstage import ChatUpstage, UpstageEmbeddings
import nest_asyncio
import os
import time

# 1. 환경 설정
nest_asyncio.apply()
load_dotenv()

# 로거 설정
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
console_handler = logging.StreamHandler()
logger.addHandler(console_handler)

# Upstage 모델 설정 
model = ChatUpstage(model="solar-pro", temperature=0.0) 
embeddings = UpstageEmbeddings(model="solar-embedding-1-large")

# 2. 파일 로드 및 컬럼 설정
input_file = "GQ_Evaluate_with_Ragas.xlsx" 
output_file = "[UpS]GQ_Finaltest_Ragas.xlsx"

# 엑셀의 컬럼 이름 지정
col_question = 'Question'         
col_ground_truth = 'Chatbot_Summary'    
col_chatbot_answer = 'AI_Answer1' 

print(f"📂 파일 로딩 중: {input_file}")
df = pd.read_excel(input_file)

# 데이터 검증
required_cols = [col_question, col_ground_truth, col_chatbot_answer]
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"❌ 엑셀 파일에 '{col}' 컬럼이 없습니다. 컬럼명을 확인해주세요.")

# 3. 배치 처리 설정
BATCH_SIZE = 10
total_rows = len(df)
processed_results = [] 

print(f"🚀 총 {total_rows}개의 데이터를 {BATCH_SIZE}개씩 나누어 평가합니다.")
print(f"📊 사용 지표: Answer Similarity, Answer Correctness")

# 4. 메인 평가 루프
for start_idx in range(0, total_rows, BATCH_SIZE):
    end_idx = min(start_idx + BATCH_SIZE, total_rows)
    print(f"\n🔄 Processing Batch: {start_idx} ~ {end_idx} (Total: {total_rows})")
    
    batch_df = df.iloc[start_idx:end_idx].copy()
    
    questions = batch_df[col_question].fillna("").astype(str).tolist()
    ground_truths = batch_df[col_ground_truth].fillna("").astype(str).tolist()
    answers = batch_df[col_chatbot_answer].fillna("").astype(str).tolist()
    contexts = [[gt] for gt in ground_truths] 
    
    data_dict = {
        "question": questions,
        "ground_truth": ground_truths,
        "answer": answers,
        "contexts": contexts
    }
    dataset = Dataset.from_dict(data_dict)
    
    try:
        results = evaluate(
            dataset=dataset,
            metrics=[answer_similarity, answer_correctness],
            llm=model,
            embeddings=embeddings,
            raise_exceptions=False 
        )
        
        res_df = results.to_pandas()
        
        # [디버깅] 실제로 생성된 컬럼명 확인
        print(f"   🔍 Result Columns: {res_df.columns.tolist()}")

        # 1. Similarity 점수
        if 'answer_similarity' in res_df.columns:
            batch_df['Score_Similarity'] = res_df['answer_similarity'].values
        else:
            batch_df['Score_Similarity'] = 0
            
        # 2. Correctness 점수
        if 'answer_correctness' in res_df.columns:
            batch_df['Score_Correctness'] = res_df['answer_correctness'].values
        else:
            batch_df['Score_Correctness'] = 0

        processed_results.append(batch_df)
        
        # 중간 저장
        all_results = pd.concat(processed_results, axis=0)
        all_results.to_excel(output_file, index=False)
        print(f"   💾 중간 저장 완료: {end_idx}행까지 -> {output_file}")
        
    except Exception as e:
        print(f"❌ [ERROR] Batch {start_idx}~{end_idx} 처리 중 오류 발생: {e}")
        print("🚨 진행된 부분까지 저장하고 멈춥니다.")
        break

print("\n✨ 모든 평가 작업 종료.")

📂 파일 로딩 중: GQ_Evaluate_with_Ragas.xlsx
🚀 총 2개의 데이터를 10개씩 나누어 평가합니다.
📊 사용 지표: Answer Similarity, Answer Correctness

🔄 Processing Batch: 0 ~ 2 (Total: 2)


Evaluating: 100%|██████████| 4/4 [00:17<00:00,  4.30s/it]


   🔍 Result Columns: ['user_input', 'retrieved_contexts', 'response', 'reference', 'answer_similarity', 'answer_correctness']
   💾 중간 저장 완료: 2행까지 -> [UpS]GQ_Finaltest_Ragas.xlsx

✨ 모든 평가 작업 종료.
